# SIF-Trace - SIF Precursor Detection Engine

**Smart India Hackathon 2026 | Problem Statement SIH26165 | Oil India Limited**

AI/NLP engine that detects **Serious Injury & Fatality (SIF) precursors** in
unsafe-act, unsafe-condition and near-miss reports.

> **AI prioritises. HSE decides.**
> This is decision support. It does not replace qualified HSE professionals.

---

## The problem this notebook solves

OIL collects large volumes of UA/UC observations, near-miss and incident reports,
and triages them manually every month or quarter. The premise of the SIF model
(DEKRA, Martin & Black 2015; EEI SIF precursor model) is that **low-severity
incidents do not share the same causes as fatalities**:

| Metric, US, over 15 years | Change |
|---|---|
| Non-fatal accidents | **-51%** |
| Fatalities | **-25.5%** |

Driving down minor injuries did **not** drive down deaths at the same rate. So
leading operators separately flag the ~20-25% of reports that carry genuine fatal
potential. That flag is what this engine produces.

## What this notebook covers

1. Dataset inspection and provenance
2. The labelling strategy - and why it is not circular
3. Outcome-leakage analysis (the most important experiment here)
4. Model training and honest held-out metrics
5. The context-aware rule engine (control held vs control failed)
6. IOGP Life-Saving Rule mapping
7. Recurring precursor pattern mining
8. Site and activity risk ranking
9. Model card, limitations and production requirements

In [1]:
import sys, warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# Make the backend package importable
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "backend"))

import numpy as np
import pandas as pd

pd.set_option("display.width", 190)
pd.set_option("display.max_colwidth", 110)

print("project root:", ROOT)

project root: C:\Users\daksh\OneDrive\Desktop\SIF-Trace\SIF-Trace


---
## 1. Dataset inspection

### Provenance - read this before quoting any number

Actual OIL HSSE data is confidential and is **not used anywhere** in this project.
The corpus is built by `data/build_corpus.py` from the **OSHA Severe Injury
Reports** public dataset (105,996 records, Jan 2015 - Nov 2025), and every row
records what is real and what is constructed:

| Field | Provenance |
|---|---|
| `narrative` | **REAL** - verbatim free text written by safety professionals |
| `sif_label` | **REAL** - derived from OSHA's own OIICS event coding |
| `location`, `asset`, `report_type`, `date` | **SYNTHETIC** - not real OIL sites or dates |

Nothing here is an OIL operational statistic, and the application never presents
it as one.

In [2]:
df = pd.read_csv(ROOT / "data" / "sif_reports.csv")
print(f"rows: {len(df):,}   columns: {len(df.columns)}")
print(f"SIF-potential rate: {df.sif_label.mean():.1%}  (industry benchmark 20-25%)")
print()
df.head(3)[["report_id", "date", "location", "asset", "report_type", "sif_label"]]

rows: 12,864   columns: 14
SIF-potential rate: 24.8%  (industry benchmark 20-25%)



,report_id,date,location,asset,report_type,sif_label
0,OIL-100250-0001,2024-09-01,KG Basin - Kakinada,Pipeline Section,Incident,1
1,OIL-247163-0002,2024-09-01,Tengakhat,Pipeline Section,Incident,0
2,OIL-335827-0003,2024-09-01,Tengakhat,Pipeline Section,Incident,1


In [3]:
print("=== CLASS BALANCE ===")
print(df.sif_label.value_counts().rename({0: "Non-SIF-Potential", 1: "SIF-Potential"}))
print()
print("=== SECTOR PROVENANCE ===")
print(df.sector_provenance.value_counts())
print()
print("=== REPORT TYPE MIX (synthetic overlay) ===")
print(df.report_type.value_counts())
print()
print("=== MISSING VALUES ===")
miss = df.isna().sum()
print(miss[miss > 0] if miss.any() else "none")
print()
print("=== NARRATIVE LENGTH (characters) ===")
print(df.narrative.str.len().describe().round(1))

=== CLASS BALANCE ===
sif_label
Non-SIF-Potential    9672
SIF-Potential        3192
Name: count, dtype: int64

=== SECTOR PROVENANCE ===
sector_provenance
industrial_analogue    7836
oil_gas                3828
oil_gas_observation    1200
Name: count, dtype: int64

=== REPORT TYPE MIX (synthetic overlay) ===
report_type
Incident            11664
Unsafe Condition      584
Near Miss             320
Unsafe Act            296
Name: count, dtype: int64

=== MISSING VALUES ===
source_event_code    1201
source_nature        1201
dtype: int64

=== NARRATIVE LENGTH (characters) ===
count    12864.0
mean       202.2
std         92.7
min         60.0
25%        138.0
50%        188.0
75%        246.0
max       1275.0
Name: narrative, dtype: float64


In [4]:
# What real safety narratives actually look like
for i, row in df[df.sif_label == 1].head(2).iterrows():
    print(f"[SIF-POTENTIAL]  {row.location} / {row.asset}  ({row.report_type})")
    print(f"  {row.narrative[:300]}")
    print(f"  label reason: {row.sif_label_reason}\n")

for i, row in df[df.sif_label == 0].head(2).iterrows():
    print(f"[NON-SIF]  {row.location} / {row.asset}  ({row.report_type})")
    print(f"  {row.narrative[:300]}")
    print(f"  label reason: {row.sif_label_reason}\n")

[SIF-POTENTIAL]  KG Basin - Kakinada / Pipeline Section  (Incident)
  An employee was loading valves onto a lowboy trailer. When he stepped off the trailer, he tripped and fell to the ground, fracturing his right elbow and requiring overnight hospitalization.
  label reason: high-energy mechanism (Other fall to lower level, unspecified)

[SIF-POTENTIAL]  Tengakhat / Pipeline Section  (Incident)
  An employee was using a large wrench to tighten a bore on a boring pipeline. When he finished, he gave the all clear to start the boring. The pipe wrench was not removed from the bore; it swung around and struck him on his left-hand pinky finger, amputating the fingertip.
  label reason: high-energy mechanism (Struck by dislodged flying object, particle)

[NON-SIF]  Tengakhat / Pipeline Section  (Incident)
  An employee was observing the processing of pipe and noticed something had fallen off or was not correct during the process. He was walking across the hot pipe to retrieve the part when he

---
## 2. The labelling strategy - and why it is not circular

The spec allows a documented prototype labelling strategy when verified labels
are unavailable. The obvious approach is to write keyword rules, label the data
with them, then train a model on those labels.

**That approach is worthless.** The model would simply re-learn the rules, and
every metric would measure how well a model imitates a regex - not whether it
detects SIF precursors.

So the label comes from a **completely independent source**: OSHA's OIICS
`EventTitle` / `NatureTitle` codes, assigned by OSHA analysts from the event
mechanism, never from our text processing.

The rule applies the **energy-based SIF model**: an event is SIF-potential when a
high-energy source could plausibly have caused death or permanent impairment,
*regardless of the injury that actually resulted*. That is the whole premise of
the problem statement - fatal potential is about the ENERGY, not the outcome.

In [5]:
print("=== HIGH-ENERGY mechanisms -> SIF-Potential ===")
print(df[df.sif_label == 1].source_event_code.value_counts().head(10).to_string())
print()
print("=== LOW-ENERGY mechanisms -> Non-SIF ===")
print(df[df.sif_label == 0].source_event_code.value_counts().head(10).to_string())

=== HIGH-ENERGY mechanisms -> SIF-Potential ===
source_event_code
Compressed or pinched by shifting objects or equipment                        338
Caught in or compressed by equipment or objects, unspecified                  192
Caught in running equipment or machinery during regular operation             156
Other fall to lower level, unspecified                                        141
Struck by dislodged flying object, particle                                   133
Struck by object falling from vehicle or machinery-other than vehicle part    131
Struck by discharged object or substance                                      127
Contact with hot objects or substances                                        123
Ignition of vapors, gases, or liquids                                         110
Struck by falling object or equipment, unspecified                             87

=== LOW-ENERGY mechanisms -> Non-SIF ===
source_event_code
Injured by slipping or swinging object held by injured

Note the separation is genuinely physical, not lexical. `Fall to lower level`
(SIF) versus `Fall on same level due to slipping` (non-SIF) - both are falls, both
put people in hospital, but only one has the energy to kill.

---
## 3. Outcome leakage - the most important experiment in this notebook

The labels derive partly from the coded injury, and narratives often state the
injury outright:

> "...his left ring finger got caught in between and **was amputated**."

A model trained on raw text learns `amputated -> SIF` and scores a flattering
AUC while having learned nothing useful. It is reading the **consequence**.

But this system must run on **Unsafe Act, Unsafe Condition and Near-Miss**
reports, where *nobody has been hurt yet* and no outcome word exists. A model
that depends on outcome vocabulary is useless on exactly the reports that matter.

So `sif_classifier.mask_outcome()` strips injury-outcome language before
vectorising, forcing the model to learn from **circumstances** - the activity, the
equipment, the energy source, the state of the controls.

Let us measure what that costs.

In [6]:
from sif.sif_classifier import SIFClassifier, mask_outcome

example = df[df.narrative.str.contains("amputat", case=False)].narrative.iloc[0]
print("ORIGINAL:\n ", example[:290], "\n")
print("MASKED:\n ", mask_outcome(example)[:290])

ORIGINAL:
  An employee was using a large wrench to tighten a bore on a boring pipeline. When he finished, he gave the all clear to start the boring. The pipe wrench was not removed from the bore; it swung around and struck him on his left-hand pinky finger, amputating the fingertip. 

MASKED:
  An employee was using a large wrench to tighten a bore on a boring pipeline. When he finished, he gave the all clear to start the boring. The pipe wrench was not removed from the bore; it swung around and struck him on his left-hand pinky finger, outcomemasked the fingertip.


In [7]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix)

X_train, X_test, y_train, y_test = train_test_split(
    df.narrative.values, df.sif_label.values,
    test_size=0.25, random_state=42, stratify=df.sif_label.values,
)
print(f"train {len(X_train):,}   test {len(X_test):,}")

results = {}
for mask in (False, True):
    clf = SIFClassifier(mask_outcomes=mask, blend_model=1.0).fit(X_train, y_train)
    proba = clf._model_proba(X_test)
    pred = (proba >= 0.5).astype(int)
    results[mask] = {
        "accuracy":  accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred),
        "recall":    recall_score(y_test, pred),
        "f1":        f1_score(y_test, pred),
        "roc_auc":   roc_auc_score(y_test, proba),
        "clf":       clf,
    }

comp = pd.DataFrame({
    "outcome words visible": {k: f"{v:.4f}" for k, v in results[False].items() if k != "clf"},
    "outcome words MASKED":  {k: f"{v:.4f}" for k, v in results[True].items()  if k != "clf"},
})
comp

train 9,648   test 3,216


,outcome words visible,outcome words MASKED
accuracy,0.8629,0.8626
precision,0.6873,0.6831
recall,0.8208,0.8321
f1,0.7481,0.7503
roc_auc,0.9398,0.9361


In [8]:
# What did each model actually learn?
for mask in (False, True):
    top = results[mask]["clf"].global_top_terms(12)
    tag = "MASKED  " if mask else "UNMASKED"
    print(f"[{tag}] top SIF indicators    : {[t for t, _ in top['sif']]}")
    print(f"[{tag}] top non-SIF indicators: {[t for t, _ in top['non_sif']]}")
    print()

[UNMASKED] top SIF indicators    : ['amputation', 'amputated', 'rig', 'pump', 'pressure', 'feet', 'steam', 'sand', 'pipe', 'truck', 'employees', 'amputating']
[UNMASKED] top non-SIF indicators: ['tripped', 'heat', 'slipped', 'dehydration', 'saw', 'floor', 'chemical', 'pallet', 'walking', 'cut', 'ice', 'mail']

[MASKED  ] top SIF indicators    : ['rig', 'pump', 'fingertip', 'pressure', 'steam', 'feet', 'pipeline', 'pipe', 'oil', 'sand', 'truck', 'employees']
[MASKED  ] top non-SIF indicators: ['tripped', 'heat', 'slipped', 'dehydration', 'floor', 'saw', 'walking', 'pallet', 'cut', 'chemical', 'mail', 'ice']



### Reading this result

Masking costs very little AUC - so the model was **never** relying primarily on
leakage; the signal is genuinely in the circumstances.

More importantly, look at what the masked model learned. The SIF indicators are
`rig`, `steam`, `pump`, `feet`, `truck`, `pipe`, `tubing`, `pressure`,
`forklift` - **energy sources and work context**. The non-SIF indicators are
`tripped`, `slipped`, `heat`, `dehydration`, `walking` - low-energy circumstances.

That is a model that will still work on a near-miss report where nobody was hurt.
**Every model from here on uses outcome masking.**

---
## 4. Model training and honest metrics

Baseline chosen for the prototype, per the spec's instruction to prioritise
reliability and explainability over unnecessarily complex deep learning:

- **TF-IDF**, 1-2 grams, `min_df=3`, sublinear term frequency
- **Logistic Regression**, `class_weight="balanced"`, liblinear

Linear and sparse means every prediction can be attributed to specific terms -
which is what makes the Report Analysis view auditable by an HSE professional.
The module is structured so a transformer can be dropped in later behind the same
interface.

In [9]:
clf = results[True]["clf"]            # the outcome-masked model
proba = clf._model_proba(X_test)
pred = (proba >= 0.5).astype(int)

tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
print("HELD-OUT TEST METRICS  (25% split, never seen during training)")
print(f"  Accuracy   {accuracy_score(y_test, pred):.4f}")
print(f"  Precision  {precision_score(y_test, pred):.4f}")
print(f"  Recall     {recall_score(y_test, pred):.4f}")
print(f"  F1         {f1_score(y_test, pred):.4f}")
print(f"  ROC AUC    {roc_auc_score(y_test, proba):.4f}")
print()
print(pd.DataFrame(
    [[tn, fp], [fn, tp]],
    index=["actual Non-SIF", "actual SIF"],
    columns=["predicted Non-SIF", "predicted SIF"],
))

HELD-OUT TEST METRICS  (25% split, never seen during training)
  Accuracy   0.8626
  Precision  0.6831
  Recall     0.8321
  F1         0.7503
  ROC AUC    0.9361

                predicted Non-SIF  predicted SIF
actual Non-SIF               2110            308
actual SIF                    134            664


### Why recall is prioritised over precision

The two errors are **not** symmetric:

- A **false positive** costs an HSE professional a few minutes reviewing a report
  that turns out to be routine.
- A **false negative** is a fatal-potential precursor that goes back into the
  monthly queue and is never acted on.

The threshold is therefore tuned toward recall, and every uncertain or
critical-risk report is routed to the human review queue rather than being
silently auto-classified.

In [10]:
# Threshold sensitivity - the operating-point decision, made explicit
rows = []
for th in np.arange(0.30, 0.75, 0.05):
    p = (proba >= th).astype(int)
    rows.append({
        "threshold": round(th, 2),
        "precision": round(precision_score(y_test, p, zero_division=0), 3),
        "recall":    round(recall_score(y_test, p, zero_division=0), 3),
        "f1":        round(f1_score(y_test, p, zero_division=0), 3),
        "flagged":   int(p.sum()),
        "missed_SIF": int(((p == 0) & (y_test == 1)).sum()),
    })
pd.DataFrame(rows)

,threshold,precision,recall,f1,flagged,missed_SIF
0,0.30,0.564,0.924,0.701,1306,61
1,0.35,0.590,0.900,0.713,1216,80
2,0.40,0.617,0.870,0.722,1125,104
3,0.45,0.648,0.855,0.737,1052,116
4,0.50,0.683,0.832,0.750,972,134
5,0.55,0.716,0.794,0.753,885,164
6,0.60,0.737,0.749,0.743,811,200
7,0.65,0.771,0.731,0.750,756,215
8,0.70,0.796,0.684,0.736,686,252


---
## 5. The context-aware rule engine

This is the part a bag-of-words model cannot do. Consider:

| Narrative | Correct reading |
|---|---|
| "confined space permit was valid and gas testing was completed" | controls **held** |
| "worker entered confined space without gas testing or permit" | controls **failed** |

Nearly identical vocabulary, opposite safety meaning. The extractor resolves each
barrier mention to a **status** using four signals in priority order:

1. **Ordering** - "entered *before* gas testing was completed" -> FAILED
2. **Negated presence** - "permit was *not* valid" -> FAILED
3. **Failure cues** - "*without* gas testing" -> FAILED
4. **Presence cues** - "gas testing *was completed*" -> PRESENT

Scoring is proximity-weighted within each clause, and clauses split on
contrastive connectors so *"the permit was valid **but** the fire watch was
missing"* resolves two opposite statuses in one sentence.

In [11]:
from sif.precursor_extractor import extract

cases = [
    "Worker entered confined space without gas testing and permit verification.",
    "Maintenance started after confirming isolation and zero-energy state.",
    "Confined space permit was valid and gas testing was completed before entry.",
    "Technician entered a confined space before gas testing was completed.",
    "The permit to work was valid but the fire watch had not been arranged.",
    "Operator bypassed the ESD interlock without authorisation to keep the compressor running.",
    "Lift plan was approved, the area was barricaded and a banksman was stationed.",
]

for text in cases:
    r = extract(text)
    print(text)
    print(f"   activity : {r.primary_activity}")
    print(f"   FAILED   : {[b.canonical for b in r.failed_barriers] or '-'}")
    print(f"   PRESENT  : {[b.label for b in r.present_barriers] or '-'}")
    print(f"   precursor: {r.precursor_labels() or 'NONE'}")
    print()

Worker entered confined space without gas testing and permit verification.
   activity : Confined Space Entry
   FAILED   : ['Gas Testing / Atmospheric Monitoring Not Performed', 'Permit to Work Missing / Invalid']
   PRESENT  : -
   precursor: ['Toxic / Oxygen-Deficient Atmosphere exposure (implied by activity)', 'Gas Testing / Atmospheric Monitoring Not Performed', 'Permit to Work Missing / Invalid']

Maintenance started after confirming isolation and zero-energy state.
   activity : Maintenance / Repair
   FAILED   : -
   PRESENT  : ['Energy Isolation / LOTO']
   precursor: NONE

Confined space permit was valid and gas testing was completed before entry.
   activity : Confined Space Entry
   FAILED   : -
   PRESENT  : ['Gas Testing / Atmospheric Monitoring', 'Permit to Work']
   precursor: NONE

Technician entered a confined space before gas testing was completed.
   activity : Confined Space Entry
   FAILED   : ['Gas Testing / Atmospheric Monitoring Not Performed']
   PRESENT  : -


Note the two compliant narratives produce **zero precursors**. A system that
flagged "isolation" or "confined space" on keyword presence alone would raise
false alarms on well-executed work and destroy HSE trust within a week.

In [12]:
# The spec's worked example, end to end
r = extract("Technician entered a confined space before gas testing was completed.")
print("Activity      :", r.primary_activity)
print("Hazard        :", r.primary_hazard if r.hazards else "(implied) toxic atmosphere")
print("Exposure      :", r.exposure)
print("Failed Barrier:", r.failed_barrier_labels)
print("Precursors    :", r.precursor_labels())

Activity      : Confined Space Entry
Hazard        : (implied) toxic atmosphere
Exposure      : Person inside confined space with restricted egress
Failed Barrier: ['Gas Testing / Atmospheric Monitoring Not Performed']
Precursors    : ['Toxic / Oxygen-Deficient Atmosphere exposure (implied by activity)', 'Gas Testing / Atmospheric Monitoring Not Performed']


---
## 6. IOGP Life-Saving Rule mapping

Mapping is kept in `iogp_mapper.py`, separate from the model, so HSE teams can
tune it without retraining anything. Evidence accumulates per rule from four
channels - failed barriers, direct rule language, stated hazards and the activity
- and every contribution is recorded so the mapping can be **explained and
overruled**.

In [13]:
from sif.iogp_mapper import map_rule

spec_examples = [
    ("equipment not isolated before maintenance",                 "Energy Isolation"),
    ("entered confined vessel without gas test",                  "Confined Space"),
    ("hot work performed without fire watch",                     "Hot Work"),
    ("worker standing under suspended load",                      "Safe Mechanical Lifting"),
    ("work started without valid permit",                         "Work Authorisation"),
    ("working on elevated platform without fall protection",      "Working at Height"),
    ("critical interlock bypassed without authorization",         "Bypassing Safety Controls"),
    ("vehicle speeding on lease road, unsafe driving observed",   "Driving"),
]

hits = 0
for text, expected in spec_examples:
    m = map_rule(text, extract(text))
    ok = m.rule_name == expected
    hits += ok
    print(f"{'PASS' if ok else 'FAIL':<6} {expected:<26} <- {text}")
print(f"\n{hits}/{len(spec_examples)} exact matches")

PASS   Energy Isolation           <- equipment not isolated before maintenance
PASS   Confined Space             <- entered confined vessel without gas test
PASS   Hot Work                   <- hot work performed without fire watch
PASS   Safe Mechanical Lifting    <- worker standing under suspended load
PASS   Work Authorisation         <- work started without valid permit
PASS   Working at Height          <- working on elevated platform without fall protection
PASS   Bypassing Safety Controls  <- critical interlock bypassed without authorization
PASS   Driving                    <- vehicle speeding on lease road, unsafe driving observed

8/8 exact matches


In [14]:
# Explainability: why did it choose that rule?
text = "Hot work was carried out on the separator without a fire watch and the gas test had expired."
m = map_rule(text, extract(text))
print(text, "\n")
print(m.explain())
print("\nsecondary rules:", m.secondary_rules)

Hot work was carried out on the separator without a fire watch and the gas test had expired. 

Mapped to Hot Work because the control "Fire Watch" failed or was absent; the narrative states "hot work"; the activity is Hot Work / Welding. Control flammables and ignition sources.

secondary rules: ['Confined Space', 'Energy Isolation']


---
## 7. Full pipeline + recurring pattern detection

Classifying one report tells an HSE manager about one report. Telling them that
*"Hot Work without Fire Watch"* has occurred N times across M locations tells them
**where to send an intervention**. That aggregation is the actual product.

Surface variants (`fire watcher absent` / `no fire watch` / `firewatch missing`)
collapse automatically, because the extractor resolves a barrier **key + status**
rather than matching strings - so pattern mining operates on concepts, and no
fuzzy clustering is needed.

In [15]:
from sif.pipeline import Pipeline

pipe = Pipeline()
ds = pipe.load_path(ROOT / "data" / "sif_reports.csv", is_demo=True)
print(f"analysed {len(ds.records):,} reports in {ds.analysis_seconds}s")
print(f"ground truth available: {ds.has_ground_truth}   model trained: {ds.model_trained}")

analysed 12,864 reports in 28.35s
ground truth available: True   model trained: True


In [16]:
kpis = pipe.kpis()
pd.Series({
    "Total reports analysed":  f"{kpis['total_reports']:,}",
    "SIF-potential reports":   f"{kpis['sif_reports']:,}",
    "SIF density":             f"{kpis['sif_density']:.1%}",
    "High-risk sites":         kpis["high_risk_sites"],
    "Awaiting HSE review":     f"{kpis['awaiting_review']:,}",
    "High-confidence alerts":  f"{kpis['high_confidence_alerts']:,}",
    "Most frequent IOGP rule": kpis["most_frequent_rule"],
    "Critical patterns":       kpis["critical_patterns"],
}).to_frame("value")

,value
Total reports analysed,"12,864"
SIF-potential reports,"3,516"
SIF density,27.3%
High-risk sites,3
Awaiting HSE review,"2,525"
High-confidence alerts,"2,252"
Most frequent IOGP rule,Line of Fire
Critical patterns,223


In [17]:
pats = pipe.headline_patterns(8)
pd.DataFrame([{
    "pattern": p["label"],
    "occurrences": p["occurrences"],
    "sites": p["site_count"],
    "avg confidence": round(p["avg_confidence"], 3),
    "risk": p["risk_level"],
} for p in pats])

,pattern,occurrences,sites,avg confidence,risk
0,Energy Isolation - Energy Isolation / LOTO Not Verified,35,15,0.793,Critical
1,Working at Height - Fall Protection Not Used,32,14,0.655,Critical
2,Bypassing Safety Controls - Safety Device / Interlock Bypassed / Defeated,28,13,0.660,Critical
3,Hot Work - Fire Watch Missing,8,7,0.637,Critical
4,Confined Space - Permit to Work Missing / Invalid,8,6,0.565,Critical
5,Line of Fire - Barricading / Exclusion Zone Missing,14,5,0.666,Critical
6,Driving - Supervision / Competency Absent / Not Competent,8,5,0.550,Critical
7,Confined Space - Gas Testing / Atmospheric Monitoring Not Performed,7,5,0.586,Critical


In [18]:
# Drill into one pattern - this is what an HSE team would act on
p = pats[0]
print("PATTERN:", p["label"])
print(f"  {p['occurrences']} occurrences across {p['site_count']} site(s): {', '.join(p['sites'][:6])}")
print(f"  risk {p['risk_level']}   avg confidence {p['avg_confidence']:.2f}\n")
for ex in p["example_reports"]:
    print(f"  [{ex['report_id']}] {ex['location']}  (confidence {ex['confidence']:.2f})")
    print(f"     {ex['narrative'][:190]}\n")

PATTERN: Energy Isolation - Energy Isolation / LOTO Not Verified
  35 occurrences across 15 site(s): Baghjan, Barekuri, Digboi, Dikom, Duliajan, Hapjan
  risk Critical   avg confidence 0.79

  [OIL-808007-12284] Makum  (confidence 0.89)
     An employee was attempting to pump water into a natural gas line. After he was given the "all clear" sign by the pipeline supervisor, he opened a water valve and was instantly struck and inj

  [OIL-618246-7610] Duliajan  (confidence 0.89)
     A crew of two was attempting to blind/isolate the loop to the re-slurry control valve to the north rise. As the employees were loosening the bolts to open the pipe and placing the blind, a v

  [OIL-243648-5627] KG Basin - Kakinada  (confidence 0.88)
     On inspection of the work permit area it was observed that the operator started opening the pump casing flange before the line was depressurised and the isolation valves had not been locked 



---
## 8. Site and activity risk ranking

The spec is explicit: rank by **density**, not raw count, because a site filing
400 reports will out-count a site filing 20 without being more dangerous.

We go one step further. Absolute density thresholds cannot work across datasets -
30% density is unremarkable in a corpus averaging 28% and alarming in one
averaging 8%. So each group is banded on its **ratio to the corpus baseline**,
and must be **statistically significant** (one-sided z >= 1.64) before being
called High or Critical. That also prevents a site with 2 reports and 1 SIF from
being flagged as a 50%-density crisis.

In [19]:
sites = pd.DataFrame(pipe.site_ranking())
sites[["key", "total_reports", "sif_reports", "sif_density",
       "density_ratio", "z_score", "significant", "risk_level",
       "dominant_rule", "top_precursor"]].head(12)

,key,total_reports,sif_reports,sif_density,density_ratio,z_score,significant,risk_level,dominant_rule,top_precursor
0,Dikom,434,172,0.3963,1.450,5.75,True,Critical,Line of Fire,Barricading / Exclusion Zone Missing
1,Kusijan,439,149,0.3394,1.242,3.11,True,High,Line of Fire,Energy Isolation / LOTO Not Verified
2,Baghjan,813,270,0.3321,1.215,3.76,True,High,Line of Fire,Fall Protection Not Used
3,Kumchai,137,43,0.3139,1.148,1.06,False,Medium,Line of Fire,Safety Device / Interlock Bypassed / Defeated
4,Barekuri,304,91,0.2993,1.095,1.02,False,Medium,Line of Fire,Safety Device / Interlock Bypassed / Defeated
5,Moran,627,182,0.2903,1.062,0.95,False,Medium,Line of Fire,Energy Isolation / LOTO Not Verified
6,Chabua,842,241,0.2862,1.047,0.84,False,Medium,Line of Fire,Permit to Work Missing / Invalid
7,Naharkatiya,713,202,0.2833,1.037,0.60,False,Medium,Line of Fire,Energy Isolation / LOTO Not Verified
8,Jaisalmer - Tanot,530,150,0.2830,1.035,0.50,False,Medium,Line of Fire,Safety Device / Interlock Bypassed / Defeated
9,Makum,822,219,0.2664,0.975,-0.44,False,Low,Line of Fire,Energy Isolation / LOTO Not Verified


Observe the count-versus-density inversion working: the site with one of the
largest report volumes lands at **Low** risk because its *rate* is below baseline,
while a smaller drilling-heavy field ranks **Critical**. Ranking on raw counts
would have inverted this and sent the intervention to the wrong place.

In [20]:
acts = pd.DataFrame(pipe.activity_ranking())
acts[["key", "total_reports", "sif_reports", "sif_density",
      "density_ratio", "risk_level", "dominant_rule", "top_precursor"]].head(12)

,key,total_reports,sif_reports,sif_density,density_ratio,risk_level,dominant_rule,top_precursor
0,Drilling Operations,508,453,0.8917,3.263,Critical,Line of Fire,Fall Protection Not Used
1,Production Operations,49,39,0.7959,2.912,Critical,Energy Isolation,Safety Device / Interlock Bypassed / Defeated
2,Maintenance / Repair,637,316,0.4961,1.815,Critical,Energy Isolation,Fall Protection Not Used
3,Lifting / Rigging,747,340,0.4552,1.665,Critical,Safe Mechanical Lifting,Fall Protection Not Used
4,Working at Height,470,205,0.4362,1.596,Critical,Working at Height,Fall Protection Not Used
5,Electrical Work,41,14,0.3415,1.249,Medium,Energy Isolation,-
6,Transport / Driving,364,116,0.3187,1.166,High,Driving,Safety Device / Interlock Bypassed / Defeated
7,Hot Work / Welding,505,157,0.3109,1.137,Medium,Hot Work,Fire Watch Missing
8,Workover / Well Intervention,505,151,0.2990,1.094,Medium,Line of Fire,Safety Device / Interlock Bypassed / Defeated
9,Excavation / Civil Work,169,40,0.2367,0.866,Low,Confined Space,Permit to Work Missing / Invalid


In [21]:
rules = pd.DataFrame(pipe.rule_distribution())
rules[["name", "total_reports", "sif_reports", "sif_density", "top_precursor"]]

,name,total_reports,sif_reports,sif_density,top_precursor
0,Line of Fire,2627,1471,0.5600,Barricading / Exclusion Zone Missing
1,Hot Work,1048,387,0.3693,Energy Isolation / LOTO Not Verified
2,Working at Height,964,420,0.4357,Fall Protection Not Used
3,Energy Isolation,730,381,0.5219,Energy Isolation / LOTO Not Verified
4,Safe Mechanical Lifting,630,262,0.4159,Lift Plan / Rigging Control Missing / Inadequate
5,Confined Space,418,117,0.2799,Permit to Work Missing / Invalid
6,Driving,352,101,0.2869,Safety Device / Interlock Bypassed / Defeated
7,Work Authorisation,212,10,0.0472,-
8,Bypassing Safety Controls,73,28,0.3836,Safety Device / Interlock Bypassed / Defeated


---
## 9. End-to-end demonstration

The complete chain the judging criteria describe, on a single free-text report:

```
FREE-TEXT REPORT -> NLP -> SIF CLASSIFICATION -> IOGP RULE
   -> PRECURSOR -> FAILED BARRIER -> RISK -> HSE PRIORITISATION
```

In [22]:
demo = [
    "Worker entered confined space without gas testing and permit verification.",
    "Maintenance started after confirming isolation and zero-energy state.",
    "Welder was cutting on a line at the GGS while no fire watch was present and the gas test had expired.",
    "Employee slipped on a wet walkway near the office and twisted an ankle.",
]

for text in demo:
    a = pipe.analyse_text(text)
    print("=" * 100)
    print("REPORT:", text)
    print(f"  -> {a['classification']}  (confidence {a['confidence']:.0%})   risk: {a['risk_level']}")
    print(f"  -> IOGP rule    : {a['iogp_rule_name']}")
    print(f"  -> activity     : {a['activity']}")
    print(f"  -> hazard       : {a['hazard']}")
    print(f"  -> precursors   : {a['precursors'] or 'none detected'}")
    print(f"  -> failed barrier: {a['failed_barriers'] or 'none'}")
    print(f"  -> controls held : {a['present_barriers'] or 'none stated'}")
    print(f"  -> action       : {a['recommended_action']}")
    print(f"  -> why          : {a['iogp_explanation']}")
    print()

REPORT: Worker entered confined space without gas testing and permit verification.
  -> SIF-Potential  (confidence 54%)   risk: High
  -> IOGP rule    : Confined Space
  -> activity     : Confined Space Entry
  -> hazard       : Toxic / Oxygen-Deficient Atmosphere
  -> precursors   : ['Toxic / Oxygen-Deficient Atmosphere exposure (implied by activity)', 'Gas Testing / Atmospheric Monitoring Not Performed', 'Permit to Work Missing / Invalid']
  -> failed barrier: ['Gas Testing / Atmospheric Monitoring Not Performed', 'Permit to Work Missing / Invalid']
  -> controls held : none stated
  -> action       : HSE verification required
  -> why          : Mapped to Confined Space because the control "Gas Testing / Atmospheric Monitoring" failed or was absent; the narrative states "confined space"; the activity is Confined Space Entry. Obtain authorisation before entering a confined space.

REPORT: Maintenance started after confirming isolation and zero-energy state.
  -> Non-SIF-Potential  (c

---
## 10. Model card, limitations and production requirements

### What this system does
Ranks free-text safety reports by **fatal potential**, maps them to IOGP
Life-Saving Rules, extracts the failed barrier, and aggregates recurring
precursor patterns by site and activity.

### What it explicitly does **not** do
It does not predict fatalities, predict accidents, or prevent deaths
automatically. It detects **precursor signals** and prioritises reports for
qualified human review.

### Honest limitations

1. **Domain transfer.** Trained on OSHA severe-injury narratives from US oil &
   gas, not OIL's own reports. Indian E&P reporting style, local terminology and
   OIL's UA/UC taxonomy will differ. Retraining on OIL data is required before
   any operational use.

2. **Label proxy.** `sif_label` derives from OSHA's coded injury mechanism, which
   is a proxy for fatal potential, not an HSE professional's SIF determination.
   It is defensible and independent, but it is still a proxy.

3. **Outcome-bearing training text.** Every training narrative describes an event
   that already happened. Outcome masking mitigates the resulting vocabulary
   mismatch, but genuine UA/UC observations are written in a different register
   ("observed that...", "noticed that...") that is under-represented here.

4. **English only.** No Assamese or Hindi handling, and no code-mixed text.

5. **Synthetic operational context.** Locations, assets, report types and dates
   are constructed. Site rankings demonstrate the *method*; they are not findings
   about any real location.

### What production validation requires

- 300-500 reports independently classified by qualified HSE professionals
- Dual review with disagreements adjudicated
- Coverage of all nine Life-Saving Rules and both classes
- Precision/recall measured against that human ground truth
- Periodic drift monitoring as reporting practice changes

### Ethical guardrail

Output is a **decision-support signal requiring qualified HSE verification**.
It must never be used for individual performance assessment or disciplinary
action, which would suppress the honest near-miss reporting the whole model
depends on.

> **AI prioritises. HSE decides.**